# Model Evaluation

Evaluates all trained models on the test set and exports results to JSON files.

Run this notebook once to generate evaluation data, then use `04_analysis.ipynb` for visualizations.

## 1. Setup

In [10]:
# imports
import json
from pathlib import Path
from datetime import datetime

import torch
import torch.nn.functional as F
import numpy as np
import scipy.io
from tqdm.notebook import tqdm
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# local imports
from src.dataset import (
    CompCarsDataset,
    get_val_transforms,
    get_preprocessed_val_transforms,
    get_dataloader
)
from src.models import (
    SimpleCNN,
    ResNet50Classifier,
    EfficientNetClassifier,
    ResNet50_SE,
    HierarchicalCarClassifier,
    count_parameters
)
from src.utils import get_model_for_inference, load_checkpoint, top_k_accuracy

# random seed for reproducibility - my student id
SEED = 2128831
np.random.seed(SEED)
torch.manual_seed(SEED)

In [11]:
# configuration
DATASET_PATH = 'dataset/data'
TEST_SPLIT = 'dataset/data/train_test_split/classification/test.txt'
NAME_MAPPING_FILE = 'dataset/data/misc/make_model_name.mat'
CHECKPOINT_DIR = 'checkpoints'
RESULTS_DIR = 'results'

# model configuration
NUM_CLASSES = 75
NUM_MODELS = 431
IMAGE_SIZE = 224

# dataloader configuration
BATCH_SIZE = 32
NUM_WORKERS = 4

# use pre-resized images (image_256 folder) for faster loading
USE_PREPROCESSED = True

# models to evaluate
MODELS_TO_EVALUATE = []
MODELS_TO_EVALUATE.append('simplecnn_label_smoothing')
MODELS_TO_EVALUATE.append('efficientnet_label_smoothing')
MODELS_TO_EVALUATE.append('resnet50_ce')
MODELS_TO_EVALUATE.append('resnet50_focal')
MODELS_TO_EVALUATE.append('resnet50_label_smoothing')
MODELS_TO_EVALUATE.append('resnet50_se_label_smoothing')
MODELS_TO_EVALUATE.append('hierarchical')

# display names for plots (saved to metadata for visualization notebook)
MODEL_DISPLAY_NAMES = {}
MODEL_DISPLAY_NAMES['simplecnn_label_smoothing'] = 'SimpleCNN'
MODEL_DISPLAY_NAMES['efficientnet_label_smoothing'] = 'EfficientNet-B0'
MODEL_DISPLAY_NAMES['resnet50_ce'] = 'ResNet50 (CE)'
MODEL_DISPLAY_NAMES['resnet50_focal'] = 'ResNet50 (Focal)'
MODEL_DISPLAY_NAMES['resnet50_label_smoothing'] = 'ResNet50 (LS)'
MODEL_DISPLAY_NAMES['resnet50_se_label_smoothing'] = 'ResNet50-SE'
MODEL_DISPLAY_NAMES['hierarchical'] = 'Hierarchical'

# colors for plots
MODEL_COLORS = {}
MODEL_COLORS['simplecnn_label_smoothing'] = '#e41a1c'
MODEL_COLORS['efficientnet_label_smoothing'] = '#377eb8'
MODEL_COLORS['resnet50_ce'] = '#4daf4a'
MODEL_COLORS['resnet50_focal'] = '#984ea3'
MODEL_COLORS['resnet50_label_smoothing'] = '#ff7f00'
MODEL_COLORS['resnet50_se_label_smoothing'] = '#a65628'
MODEL_COLORS['hierarchical'] = '#f781bf'

# device setup
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"using device: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("using device: cpu")

using device: NVIDIA GeForce RTX 3060


## 2. Helper Functions

In [12]:
def evaluate_model(model, dataloader, device):
    # evaluate single-task model
    
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    all_indices = []
    
    sample_idx = 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='evaluating'):
            images = images.to(device)
            
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, preds = outputs.max(1)
            
            batch_size = images.size(0)
            for i in range(batch_size):
                all_indices.append(sample_idx)
                sample_idx = sample_idx + 1
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    all_indices = np.array(all_indices)
    
    # calculate metrics
    top1_acc = accuracy_score(all_labels, all_preds)
    top5_acc = top_k_accuracy(all_probs, all_labels, k=5)
    
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average='macro',
        zero_division=0
    )
    
    results = {}
    results['predictions'] = all_preds
    results['labels'] = all_labels
    results['probs'] = all_probs
    results['indices'] = all_indices
    results['top1_accuracy'] = top1_acc
    results['top5_accuracy'] = top5_acc
    results['precision'] = precision
    results['recall'] = recall
    results['f1_score'] = f1
    
    return results


def evaluate_hierarchical_model(model, dataloader, device):
    # evaluate hierarchical model on both tasks
    
    model.eval()
    
    make_preds = []
    make_labels = []
    make_probs = []
    
    model_preds = []
    model_labels = []
    model_probs = []
    
    all_indices = []
    sample_idx = 0
    
    with torch.no_grad():
        for images, make_label, model_label in tqdm(dataloader, desc='evaluating'):
            images = images.to(device)
            
            make_logits, model_logits = model(images)
            
            make_prob = F.softmax(make_logits, dim=1)
            model_prob = F.softmax(model_logits, dim=1)
            
            _, make_pred = make_logits.max(1)
            _, model_pred = model_logits.max(1)
            
            batch_size = images.size(0)
            for i in range(batch_size):
                all_indices.append(sample_idx)
                sample_idx = sample_idx + 1
            
            make_preds.extend(make_pred.cpu().numpy())
            make_labels.extend(make_label.numpy())
            make_probs.extend(make_prob.cpu().numpy())
            
            model_preds.extend(model_pred.cpu().numpy())
            model_labels.extend(model_label.numpy())
            model_probs.extend(model_prob.cpu().numpy())
    
    # convert to numpy
    make_preds = np.array(make_preds)
    make_labels = np.array(make_labels)
    make_probs = np.array(make_probs)
    
    model_preds = np.array(model_preds)
    model_labels = np.array(model_labels)
    model_probs = np.array(model_probs)
    
    all_indices = np.array(all_indices)
    
    # calculate make metrics
    make_top1 = accuracy_score(make_labels, make_preds)
    make_top5 = top_k_accuracy(make_probs, make_labels, k=5)
    make_precision, make_recall, make_f1, _ = precision_recall_fscore_support(
        make_labels, make_preds, average='macro', zero_division=0
    )
    
    # calculate model metrics
    model_top1 = accuracy_score(model_labels, model_preds)
    model_top5 = top_k_accuracy(model_probs, model_labels, k=5)
    model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(
        model_labels, model_preds, average='macro', zero_division=0
    )
    
    results = {}
    results['indices'] = all_indices
    
    # make task data
    results['make_predictions'] = make_preds
    results['make_labels'] = make_labels
    results['make_probs'] = make_probs
    results['make_top1_accuracy'] = make_top1
    results['make_top5_accuracy'] = make_top5
    results['make_precision'] = make_precision
    results['make_recall'] = make_recall
    results['make_f1_score'] = make_f1
    
    # model task data
    results['model_predictions'] = model_preds
    results['model_labels'] = model_labels
    results['model_probs'] = model_probs
    results['model_top1_accuracy'] = model_top1
    results['model_top5_accuracy'] = model_top5
    results['model_precision'] = model_precision
    results['model_recall'] = model_recall
    results['model_f1_score'] = model_f1
    
    return results

In [13]:
def save_evaluation_data(results, model_name, num_params, is_hierarchical, save_dir):
    # save evaluation results to json
    
    model_dir = Path(save_dir) / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    
    data = {}
    data['model_name'] = model_name
    data['num_parameters'] = num_params
    data['is_hierarchical'] = is_hierarchical
    data['evaluated_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    # convert numpy arrays to lists for json serialization
    data['indices'] = results['indices'].tolist()
    
    if is_hierarchical:
        # make task
        data['make_predictions'] = results['make_predictions'].tolist()
        data['make_labels'] = results['make_labels'].tolist()
        data['make_probs'] = results['make_probs'].tolist()
        
        data['make_metrics'] = {}
        data['make_metrics']['top1_accuracy'] = float(results['make_top1_accuracy'])
        data['make_metrics']['top5_accuracy'] = float(results['make_top5_accuracy'])
        data['make_metrics']['precision'] = float(results['make_precision'])
        data['make_metrics']['recall'] = float(results['make_recall'])
        data['make_metrics']['f1_score'] = float(results['make_f1_score'])
        
        # model task
        data['model_predictions'] = results['model_predictions'].tolist()
        data['model_labels'] = results['model_labels'].tolist()
        data['model_probs'] = results['model_probs'].tolist()
        
        data['model_metrics'] = {}
        data['model_metrics']['top1_accuracy'] = float(results['model_top1_accuracy'])
        data['model_metrics']['top5_accuracy'] = float(results['model_top5_accuracy'])
        data['model_metrics']['precision'] = float(results['model_precision'])
        data['model_metrics']['recall'] = float(results['model_recall'])
        data['model_metrics']['f1_score'] = float(results['model_f1_score'])
    else:
        data['predictions'] = results['predictions'].tolist()
        data['labels'] = results['labels'].tolist()
        data['probs'] = results['probs'].tolist()
        
        data['metrics'] = {}
        data['metrics']['top1_accuracy'] = float(results['top1_accuracy'])
        data['metrics']['top5_accuracy'] = float(results['top5_accuracy'])
        data['metrics']['precision'] = float(results['precision'])
        data['metrics']['recall'] = float(results['recall'])
        data['metrics']['f1_score'] = float(results['f1_score'])
    
    save_path = model_dir / 'evaluation_data.json'
    with open(save_path, 'w') as f:
        json.dump(data, f, indent=2)
    
    print(f"saved: {save_path}")
    
    return save_path

## 3. Load Dataset

In [14]:
# load test dataset
print("loading test dataset...")

# select transforms based on preprocessing flag
if USE_PREPROCESSED:
    test_transforms = get_preprocessed_val_transforms(IMAGE_SIZE)
    print("using preprocessed images (image_256)")
else:
    test_transforms = get_val_transforms(IMAGE_SIZE)
    print("using original images")

# standard dataset for single-task models
test_dataset = CompCarsDataset(
    root_dir=DATASET_PATH,
    split_file=TEST_SPLIT,
    transform=test_transforms,
    task='make',
    use_preprocessed=USE_PREPROCESSED
)

# hierarchical dataset - use metadata for consistent label mappings with training
test_dataset_hier = CompCarsDataset(
    root_dir=DATASET_PATH,
    split_file=TEST_SPLIT,
    transform=test_transforms,
    task='hierarchical',
    use_preprocessed=USE_PREPROCESSED,
    metadata_file='dataset/processed/metadata.json'
)

print(f"test samples: {len(test_dataset)}")
print(f"number of makes: {test_dataset.num_makes}")
print(f"number of models: {test_dataset.num_models}")

loading test dataset...
using preprocessed images (image_256)
test samples: 14939
number of makes: 75
number of models: 431


In [15]:
# create dataloaders
test_loader = get_dataloader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

test_loader_hier = get_dataloader(
    test_dataset_hier,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=False
)

print(f"test batches: {len(test_loader)}")

test batches: 467


In [16]:
# save visualization metadata (class names, image paths, etc.)
# this allows 04_analysis.ipynb to run without loading dataset or models

print("saving visualization metadata...")

# load make/model names from mat file
mat = scipy.io.loadmat(NAME_MAPPING_FILE)

all_make_names = []
for i in range(len(mat['make_names'])):
    arr = mat['make_names'][i][0]
    if len(arr) > 0:
        name = str(arr[0])
    else:
        name = f'Unknown_{i+1}'
    all_make_names.append(name)

# get unique make/model IDs from split file
with open(TEST_SPLIT, 'r') as f:
    lines = f.readlines()

unique_makes = set()
unique_models = set()

for line in lines:
    parts = line.strip().split('/')
    if len(parts) >= 2:
        make_id = int(parts[0])
        model_id = int(parts[1])
        unique_makes.add(make_id)
        unique_models.add((make_id, model_id))

sorted_makes = sorted(list(unique_makes))
sorted_models = sorted(list(unique_models))

# build class idx to name mappings
make_idx_to_name = {}
for idx in range(len(sorted_makes)):
    make_id = sorted_makes[idx]
    name = all_make_names[make_id - 1]
    make_idx_to_name[str(idx)] = name  # JSON keys must be strings

model_idx_to_name = {}
for idx in range(len(sorted_models)):
    make_id = sorted_models[idx][0]
    model_id = sorted_models[idx][1]
    make_name = all_make_names[make_id - 1]
    model_name = f'{make_name}_{model_id}'
    model_idx_to_name[str(idx)] = model_name

# get image paths from dataset
image_paths = []
for p in test_dataset.image_paths:
    image_paths.append(str(p))

# save metadata
metadata = {}
metadata['make_idx_to_name'] = make_idx_to_name
metadata['model_idx_to_name'] = model_idx_to_name
metadata['image_paths'] = image_paths
metadata['models_to_analyze'] = MODELS_TO_EVALUATE
metadata['model_display_names'] = MODEL_DISPLAY_NAMES
metadata['model_colors'] = MODEL_COLORS
metadata['num_samples'] = len(test_dataset)
metadata['num_makes'] = len(make_idx_to_name)
metadata['num_models'] = len(model_idx_to_name)
metadata['created_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

metadata_path = Path(RESULTS_DIR) / 'visualization_metadata.json'
metadata_path.parent.mkdir(parents=True, exist_ok=True)

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"saved: {metadata_path}")
print(f"  make classes: {len(make_idx_to_name)}")
print(f"  model classes: {len(model_idx_to_name)}")
print(f"  image paths: {len(image_paths)}")

saving visualization metadata...
saved: results\visualization_metadata.json
  make classes: 75
  model classes: 431
  image paths: 14939


## 4. Evaluate All Models

In [17]:
# evaluate each model and save results
for model_name in MODELS_TO_EVALUATE:
    print("\n" + "=" * 60)
    print(f"evaluating: {model_name}")
    print("=" * 60)
    
    # check checkpoint exists
    checkpoint_path = Path(CHECKPOINT_DIR) / f'{model_name}_best.pth'
    if not checkpoint_path.exists():
        print(f"checkpoint not found: {checkpoint_path}")
        continue
    
    # determine if hierarchical
    is_hierarchical = 'hierarchical' in model_name
    
    # create model (using get_model_for_inference since we load weights from checkpoint)
    if is_hierarchical:
        model = get_model_for_inference(model_name, NUM_CLASSES, NUM_MODELS)
    else:
        model = get_model_for_inference(model_name, NUM_CLASSES)
    
    # load checkpoint
    model = load_checkpoint(model, checkpoint_path, DEVICE)
    model = model.to(DEVICE)
    model.eval()
    
    # count parameters
    params = count_parameters(model)
    num_params = params['total']
    print(f"parameters: {num_params:,}")
    
    # evaluate
    if is_hierarchical:
        results = evaluate_hierarchical_model(model, test_loader_hier, DEVICE)
        print(f"\nmake task:")
        print(f"  top-1: {results['make_top1_accuracy'] * 100:.2f}%")
        print(f"  top-5: {results['make_top5_accuracy'] * 100:.2f}%")
        print(f"model task:")
        print(f"  top-1: {results['model_top1_accuracy'] * 100:.2f}%")
        print(f"  top-5: {results['model_top5_accuracy'] * 100:.2f}%")
    else:
        results = evaluate_model(model, test_loader, DEVICE)
        print(f"\nresults:")
        print(f"  top-1: {results['top1_accuracy'] * 100:.2f}%")
        print(f"  top-5: {results['top5_accuracy'] * 100:.2f}%")
        print(f"  precision: {results['precision']:.4f}")
        print(f"  recall: {results['recall']:.4f}")
        print(f"  f1: {results['f1_score']:.4f}")
    
    # save evaluation data
    save_evaluation_data(results, model_name, num_params, is_hierarchical, RESULTS_DIR)
    
    # free memory
    del model
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("evaluation complete!")
print("=" * 60)


evaluating: simplecnn_label_smoothing


parameters: 1,599,051


evaluating:   0%|          | 0/467 [00:00<?, ?it/s]


results:
  top-1: 53.08%
  top-5: 77.12%
  precision: 0.5721
  recall: 0.4260
  f1: 0.4592
saved: results\simplecnn_label_smoothing\evaluation_data.json

evaluating: efficientnet_label_smoothing
parameters: 4,103,623


evaluating:   0%|          | 0/467 [00:00<?, ?it/s]

...


results:
  top-1: 82.69%
  top-5: 88.47%
  precision: 0.7941
  recall: 0.8142
  f1: 0.7699
saved: results\resnet50_label_smoothing\evaluation_data.json

evaluating: resnet50_se_label_smoothing
parameters: 24,362,107


evaluating:   0%|          | 0/467 [00:00<?, ?it/s]


results:
  top-1: 84.48%
  top-5: 93.78%
  precision: 0.9107
  recall: 0.8208
  f1: 0.8532
saved: results\resnet50_se_label_smoothing\evaluation_data.json

evaluating: hierarchical
parameters: 25,865,786


evaluating:   0%|          | 0/467 [00:00<?, ?it/s]


make task:
  top-1: 87.55%
  top-5: 94.77%
model task:
  top-1: 82.72%
  top-5: 91.80%
saved: results\hierarchical\evaluation_data.json

evaluation complete!


In [18]:
# list generated files
print("\ngenerated evaluation data files:")
for model_name in MODELS_TO_EVALUATE:
    data_path = Path(RESULTS_DIR) / model_name / 'evaluation_data.json'
    if data_path.exists():
        size_kb = data_path.stat().st_size / 1024
        print(f"  {model_name}: {size_kb:.1f} KB")

print("\nnext step: run 04_analysis.ipynb to generate visualizations")


generated evaluation data files:
  simplecnn_label_smoothing: 32967.2 KB
  efficientnet_label_smoothing: 32980.6 KB
  resnet50_ce: 31554.0 KB
  resnet50_focal: 32981.3 KB
  resnet50_label_smoothing: 32975.7 KB
  resnet50_se_label_smoothing: 32974.7 KB
  hierarchical: 223120.5 KB

next step: run 04_analysis.ipynb to generate visualizations


## 5. Inference Time Measurement

Measure forward pass time for each model to compare computational efficiency.

In [19]:
import time

def measure_inference_time(model, dataloader, device, num_warmup=10, is_hierarchical=False):
    # measure forward pass time for a model
    # returns dict with timing statistics (total_time_sec, avg_batch_time_ms, etc.)
    
    model.eval()
    
    # warmup runs (GPU needs warmup for accurate timing)
    warmup_count = 0
    with torch.no_grad():
        for images, *_ in dataloader:
            if warmup_count >= num_warmup:
                break
            images = images.to(device)
            _ = model(images)
            warmup_count = warmup_count + 1
    
    # synchronize GPU before timing
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # timed inference
    total_samples = 0
    batch_times = []
    
    with torch.no_grad():
        for images, *_ in dataloader:
            batch_size = images.size(0)
            images = images.to(device)
            
            # synchronize before timing
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            start_time = time.perf_counter()
            _ = model(images)
            
            # synchronize after forward pass
            if device.type == 'cuda':
                torch.cuda.synchronize()
            
            end_time = time.perf_counter()
            
            batch_times.append(end_time - start_time)
            total_samples = total_samples + batch_size
    
    # calculate statistics
    total_time = sum(batch_times)
    avg_batch_time = total_time / len(batch_times)
    avg_sample_time = total_time / total_samples
    throughput = total_samples / total_time
    
    result = {}
    result['total_time_sec'] = total_time
    result['avg_batch_time_ms'] = avg_batch_time * 1000
    result['avg_sample_time_ms'] = avg_sample_time * 1000
    result['throughput_images_per_sec'] = throughput
    result['num_samples'] = total_samples
    result['num_batches'] = len(batch_times)
    
    return result

In [20]:
# measure inference time for each model
print("measuring inference times...")
print(f"device: {DEVICE}")
print(f"batch size: {BATCH_SIZE}")
print()

timing_results = {}

for model_name in MODELS_TO_EVALUATE:
    print(f"timing: {MODEL_DISPLAY_NAMES[model_name]}...", end=" ")
    
    # check checkpoint exists
    checkpoint_path = Path(CHECKPOINT_DIR) / f'{model_name}_best.pth'
    if not checkpoint_path.exists():
        print("checkpoint not found, skipping")
        continue
    
    # determine if hierarchical
    is_hierarchical = 'hierarchical' in model_name
    
    # create model (using get_model_for_inference since we load weights from checkpoint)
    if is_hierarchical:
        model = get_model_for_inference(model_name, NUM_CLASSES, NUM_MODELS)
    else:
        model = get_model_for_inference(model_name, NUM_CLASSES)
    
    # load checkpoint
    model = load_checkpoint(model, checkpoint_path, DEVICE)
    model = model.to(DEVICE)
    model.eval()
    
    # count parameters
    params = count_parameters(model)
    num_params = params['total']
    
    # select dataloader
    if is_hierarchical:
        loader = test_loader_hier
    else:
        loader = test_loader
    
    # measure timing
    timing = measure_inference_time(model, loader, DEVICE, num_warmup=10, is_hierarchical=is_hierarchical)
    
    # store results
    timing_entry = {}
    timing_entry['display_name'] = MODEL_DISPLAY_NAMES[model_name]
    timing_entry['parameters'] = num_params
    timing_entry['total_time_sec'] = timing['total_time_sec']
    timing_entry['avg_batch_time_ms'] = timing['avg_batch_time_ms']
    timing_entry['avg_sample_time_ms'] = timing['avg_sample_time_ms']
    timing_entry['throughput_images_per_sec'] = timing['throughput_images_per_sec']
    timing_results[model_name] = timing_entry
    
    print(f"{timing['throughput_images_per_sec']:.1f} img/s ({timing['avg_sample_time_ms']:.2f} ms/img)")
    
    # free memory
    del model
    torch.cuda.empty_cache()

print("\ndone!")

measuring inference times...
device: cuda
batch size: 32

timing: SimpleCNN... 1733.6 img/s (0.58 ms/img)
timing: EfficientNet-B0... 844.2 img/s (1.18 ms/img)
timing: ResNet50 (CE)... 411.0 img/s (2.43 ms/img)
timing: ResNet50 (Focal)... 410.1 img/s (2.44 ms/img)
timing: ResNet50 (LS)... 406.1 img/s (2.46 ms/img)
timing: ResNet50-SE... 393.5 img/s (2.54 ms/img)
timing: Hierarchical... 419.7 img/s (2.38 ms/img)

done!


In [21]:
# save timing results to JSON
timing_save_path = Path(RESULTS_DIR) / 'comparison' / 'inference_timing.json'
timing_save_path.parent.mkdir(parents=True, exist_ok=True)

timing_data = {}
timing_data['device'] = str(DEVICE)
if torch.cuda.is_available():
    timing_data['gpu_name'] = torch.cuda.get_device_name(0)
else:
    timing_data['gpu_name'] = 'N/A'
timing_data['batch_size'] = BATCH_SIZE
timing_data['num_test_samples'] = len(test_dataset)
timing_data['measured_at'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
timing_data['models'] = timing_results

with open(timing_save_path, 'w') as f:
    json.dump(timing_data, f, indent=2)

print(f"saved: {timing_save_path}")

# display summary table
print("\n" + "=" * 80)
print("INFERENCE TIME COMPARISON")
print("=" * 80)
print(f"{'Model':<20} {'Params (M)':<12} {'ms/image':<12} {'images/sec':<12}")
print("-" * 80)

for model_name in MODELS_TO_EVALUATE:
    if model_name in timing_results:
        r = timing_results[model_name]
        params_m = r['parameters'] / 1e6
        print(f"{r['display_name']:<20} {params_m:<12.2f} {r['avg_sample_time_ms']:<12.2f} {r['throughput_images_per_sec']:<12.1f}")

print("=" * 80)

saved: results\comparison\inference_timing.json

INFERENCE TIME COMPARISON
Model                Params (M)   ms/image     images/sec  
--------------------------------------------------------------------------------
SimpleCNN            1.60         0.58         1733.6      
EfficientNet-B0      4.10         1.18         844.2       
ResNet50 (CE)        24.60        2.43         411.0       
ResNet50 (Focal)     24.60        2.44         410.1       
ResNet50 (LS)        24.60        2.46         406.1       
ResNet50-SE          24.36        2.54         393.5       
Hierarchical         25.87        2.38         419.7       
